In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.1 Coulomb's Law and the Electric Field

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.1",
    title="Coulomb's Law and the Electric Field",
    blurb="The electric field as a vector field: Coulomb's inverse-square law, "
    "superposition, field lines, and the point charge as an idealised "
    "delta-function source. The computational vocabulary the rest of "
    "the volume is written in.",
    difficulty="intermediate",
    estimate="90–120 min",
)

## Notebook overview

Volume III opens by changing the question. Mechanics tracked particles; here the
protagonist is the **field**, a quantity defined at every point of space that a
charge placed there would feel. Coulomb's inverse-square law is the seed, but the
moment there is more than one charge the interesting object is no longer a force
between two bodies: it is the whole vector field $\mathbf E(\mathbf r)$ filling
space, the superposition of every source. Learning to build, plot, and probe that
field numerically is the real subject of this notebook, and the toolbox the rest
of the volume leans on.

We will represent a field as arrays on a grid, add fields by **superposition**,
trace **field lines** by integrating a streamline, and compute the **flux** of a
field through a surface, which turns out to equal the enclosed charge over
$\varepsilon_0$ and previews Gauss's law ([§3.3](gauss-law.ipynb)). Along the way we meet the point
charge in its honest mathematical clothing, the **Dirac delta**, developed not as
an assertion but as the limit of a narrowing Gaussian. That idealisation is what
makes the Green's-function solution of Poisson's equation possible in
[§3.4](laplace-poisson.ipynb), where
$\nabla^2(1/r) = -4\pi\,\delta^3(\mathbf r)$ makes the point charge the
fundamental solution. We thread the delta through the volume rather than isolating
it.

Everything is in **SI units**: $\varepsilon_0 = 8.854\times10^{-12}\,$F/m and
$k = 1/4\pi\varepsilon_0 \approx 8.99\times10^9\,$N·m²/C². Electrostatic fields do
not move, so every figure here is a faithful *static* plot: field lines at equal
angular spacing (whose density then encodes $|\mathbf E|$), or arrows whose length
and colour encode $|\mathbf E|$. A still picture shows a vector field better than
motion would.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: an inverse-square ratio, a symmetry cancellation, a flux that
> matches $q/\varepsilon_0$, a closed-form field. A ✓ is strong evidence; a ✗ is a
> prompt to *locate the discrepancy*, not a verdict.

> **Scope.** A working review, not a full electrodynamics course. The standard
> references are Nolting, *Theoretical Physics 3* {cite}`nolting3`, Griffiths,
> *Introduction to Electrodynamics* {cite}`griffiths_em`, and Jackson,
> *Classical Electrodynamics* {cite}`jackson` for depth.

## Theory in brief

### Coulomb's law and the electric field

Two point charges $q_1, q_2$ separated by $\mathbf r$ (pointing from $q_1$ to the
field point) repel or attract along the line joining them, with a force that falls
as the inverse square of the distance:

```{math}
:label: eq-coulomb
\mathbf F = \frac{1}{4\pi\varepsilon_0}\,\frac{q_1 q_2}{r^2}\,\hat{\mathbf r},
\qquad k \equiv \frac{1}{4\pi\varepsilon_0} \approx 8.99\times10^{9}\ \text{N·m}^2/\text{C}^2 .
```

Dividing out a test charge $q_{\text{test}}$ removes it from the discussion and
leaves a property of space alone, the **electric field**:

```{math}
:label: eq-field
\mathbf E(\mathbf r) = \frac{\mathbf F}{q_{\text{test}}}
= \frac{1}{4\pi\varepsilon_0}\,\frac{q}{r^2}\,\hat{\mathbf r},
```

the field of a single point charge $q$. It is defined everywhere (except at the
charge), and a test charge dropped in merely samples it. Numerically we represent
$\mathbf E$ as a pair of component arrays evaluated on a grid.

### Superposition

Maxwell's equations are linear, so fields add. The field of a collection of
charges is the vector sum of their individual point-charge fields, and for a
continuous distribution $\rho(\mathbf r')$ the sum becomes an integral:

```{math}
:label: eq-superposition
\mathbf E(\mathbf r) = \frac{1}{4\pi\varepsilon_0}
\sum_i q_i\,\frac{\mathbf r - \mathbf r_i}{|\mathbf r - \mathbf r_i|^3}
\;\longrightarrow\;
\frac{1}{4\pi\varepsilon_0}\int \rho(\mathbf r')\,
\frac{\mathbf r - \mathbf r'}{|\mathbf r - \mathbf r'|^3}\,d^3r' .
```

Linearity is what makes electrostatics tractable: build the field of anything by
adding the fields of its parts.

### Field lines

A **field line** is a curve everywhere tangent to $\mathbf E$. Parametrising it by
arc length $s$ gives the unit-speed streamline

```{math}
:label: eq-fieldline
\frac{d\mathbf r}{ds} = \frac{\mathbf E(\mathbf r)}{|\mathbf E(\mathbf r)|},
```

which we integrate as an ODE. Field lines begin on positive charges and end on
negative ones, and their local density encodes the field strength.

### The dipole

Two opposite charges $\pm q$ separated by a small displacement $\mathbf a$ (from
$-q$ to $+q$) form a **dipole**, with dipole moment

```{math}
:label: eq-dipole
\mathbf p = q\,\mathbf a, \qquad
E_{\text{axis}} \approx \frac{1}{4\pi\varepsilon_0}\,\frac{2p}{r^3}
\quad (r \gg a).
```

The far field falls as $1/r^3$, one power faster than a single charge, because the
charges nearly cancel (Griffiths {cite}`griffiths_em`, ch. 3, carries the
expansion out in full). The *ideal* (point) dipole is the limit $a\to0$ at fixed
$p$; for finite $a$ the on-axis field carries a small correction we will see
explicitly.

### The point charge as a delta source

A point charge is an idealisation: a finite charge $q$ squeezed to a single point.
Its charge density is written with the three-dimensional **Dirac delta**,

```{math}
:label: eq-delta
\rho(\mathbf r) = q\,\delta^3(\mathbf r - \mathbf r_0),
```

which is not a function but a **distribution**, defined by its *sifting* property
$\int f(\mathbf r)\,\delta^3(\mathbf r - \mathbf r_0)\,d^3r = f(\mathbf r_0)$. One
realises it as the limit of *nascent* functions: a Gaussian of unit integral whose
width $w\to0$ so that the peak diverges while the area stays $1$. The delta is the
bridge to the Green's-function solution of Poisson's equation in
[§3.4](laplace-poisson.ipynb), where
$\nabla^2(1/r) = -4\pi\,\delta^3(\mathbf r)$ makes the point charge the
fundamental solution of electrostatics.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from ecp import draw, validate

# SI constants used throughout the volume.
from scipy.constants import epsilon_0 as EPS0  # vacuum permittivity, F/m

K = 1.0 / (4.0 * np.pi * EPS0)  # Coulomb constant ≈ 8.99e9 N·m²/C²
print(f"ε₀ = {EPS0:.4e} F/m")
print(f"k = 1/4πε₀ = {K:.4e} N·m²/C²")

# Conventional colours: positive charge red, negative charge dark blue.
POS, NEG = "#c1121f", "#16213e"


def E_point(q, r0, X, Y):
    """In-plane components of a point charge's electric field.

    The 3-D Coulomb field (eq-field in the theory section) evaluated in the
    z = 0 plane: magnitude ~ 1/r^2, direction radial. The singular value at the
    charge is left as inf/nan for the caller to mask.

    Parameters
    ----------
    q : float
        Charge in coulombs.
    r0 : tuple of float
        Source position ``(x0, y0)`` in metres.
    X, Y : numpy.ndarray
        Coordinate grids of field points.

    Returns
    -------
    Ex, Ey : numpy.ndarray
        The two in-plane field components on the grid, in V/m.
    """
    dx, dy = X - r0[0], Y - r0[1]
    d = np.sqrt(dx**2 + dy**2)
    with np.errstate(divide="ignore", invalid="ignore"):
        Ex = K * q * dx / d**3
        Ey = K * q * dy / d**3
    return Ex, Ey


def E_field(charges, X, Y):
    """Total field of several point charges by superposition (eq-superposition).

    The linearity of Coulomb's law lets us add the individual point-charge fields.

    Parameters
    ----------
    charges : list of tuple
        Each entry ``(q, (x0, y0))``: a charge in coulombs and its position in metres.
    X, Y : numpy.ndarray
        Coordinate grids of field points.

    Returns
    -------
    Ex, Ey : numpy.ndarray
        The summed in-plane field components on the grid, in V/m.
    """
    Ex = np.zeros(np.shape(X), dtype=float)
    Ey = np.zeros(np.shape(X), dtype=float)
    for q, r0 in charges:
        ex, ey = E_point(q, r0, X, Y)
        Ex, Ey = Ex + ex, Ey + ey
    return Ex, Ey


def field_grid(n=400, half=0.5):
    """A square coordinate grid centred on the origin.

    Parameters
    ----------
    n : int, optional
        Points per side (default ``400``).
    half : float, optional
        Half-width of the domain in metres (default ``0.5``), so the grid spans
        ``[-half, half]`` on each axis.

    Returns
    -------
    X, Y : numpy.ndarray
        The 2-D meshgrid coordinate arrays.
    """
    xs = np.linspace(-half, half, n)
    return np.meshgrid(xs, xs)


NANO = 1e-9  # 1 nC, the charge scale used throughout

## Exercise 1 — The field of a point charge

The whole volume rests on one field, so we
start by building it and confirming it behaves. A single charge produces a purely
radial field whose magnitude obeys the inverse-square law of {eq}`eq-coulomb` and
{eq}`eq-field`, the fingerprint we check first.

**This exercise.** Place a charge $q = 1\,$nC at the origin.

1. Compute $\mathbf E$ on a grid covering $[-0.5, 0.5]^2\,$m (the `E_point`
   helper) and draw it ({numref}`fig-coulomb-point`).
2. Sample the magnitude at $r = 0.1, 0.2, 0.4\,$m along the $x$-axis and
   verify that halving the distance quadruples the field, i.e.
   $E(0.1)/E(0.2) = 4$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(ratio_12, 4.0, "the point-charge field falls as 1/r²", rtol=1e-3)

## Exercise 2 — Superposition and symmetry

Fields add, and adding them makes symmetry
visible. With two identical charges placed symmetrically about the origin, any
point on the perpendicular bisector feels two contributions whose sideways parts
are mirror images and must cancel, leaving a purely transverse field on the axis.
This is superposition {eq}`eq-superposition` doing the bookkeeping that, by hand,
would be a vector diagram.

**This exercise.** Place $q = +1\,$nC at $(-0.1, 0)$ and $q = +1\,$nC at
$(+0.1, 0)$.

1. Compute the total field with the `E_field` superposition helper
   ({numref}`fig-coulomb-pair`).
2. Confirm that along the $y$-axis (the perpendicular bisector) the
   $x$-component vanishes by symmetry.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    Ex_axis,
    np.zeros_like(Ex_axis),
    "the transverse field cancels on the symmetry axis",
    atol=1e-9,
)

## Exercise 3 — The electric dipole

Bring two *opposite* charges close together and the
monopole fields nearly cancel, leaving a characteristic field that falls as
$1/r^3$ ({eq}`eq-dipole`). The dipole is the first term in the systematic
description of any neutral charge distribution (the multipole expansion of [§3.5](multipole-expansion.ipynb)),
so its signature is worth meeting directly.

**This exercise.** Place $+1\,$nC at $(0, +0.01)$ and $-1\,$nC at
$(0, -0.01)$, a separation $a = 0.02\,$m and dipole moment $p = qa$.

1. Compute the on-axis field from the `E_field` superposition helper at
   $z = 0.1, 0.2\,$m.
2. Confirm the far-field $1/r^3$ falloff ($E(0.1)/E(0.2) = 8$).
3. Compare the magnitude with the ideal-dipole formula $2kp/r^3$. The small
   gap is the finite-separation correction: the ideal formula is the $a\to0$
   limit.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(ratio_dip, 8.0, "the dipole far-field falls as 1/r³", rtol=0.05)

## Exercise 4 — Tracing field lines

A field map shows direction at sampled points; a
**field line** stitches those directions into a continuous curve, integrating
{eq}`eq-fieldline` from a seed point. Seeding the lines at *equal angles* around the
positive charge sends equal flux down each, so the resulting line density is itself
a faithful picture of the field strength.

**This exercise.** Around the positive charge of the dipole (Exercise 3), seed
twelve points at equal angles and integrate $d\mathbf r/ds = \mathbf E/|\mathbf E|$
with `scipy.integrate.solve_ivp` (a fine `max_step`, `rtol=1e-8`, `atol=1e-10`,
`dense_output=True`), stopping each line when it reaches the negative charge via a
terminal event. Draw the completed lines as a static diagram
({numref}`fig-coulomb-lines`): every line that leaves $+q$ arrives at $-q$.

In [ ]:
# (solution hidden on the public site)


With the lines traced, draw them as a single static diagram: each curve in full,
the charges as fixed markers.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    end_dist < 0.02,
    "a field line seeded near +q terminates near −q",
    f"line end is {end_dist:.4f} m from −q",
)

## Exercise 5 — The point charge as a delta source

A point charge holds a finite charge at a single
point, so its density {eq}`eq-delta` cannot be an ordinary function: it must be
zero everywhere yet integrate to $q$. The **Dirac delta** captures this as a
limit. The Gaussian
$g_w(x) = \exp(-x^2/2w^2)/(w\sqrt{2\pi})$ has unit integral for *every* width $w$,
and as $w\to0$ its peak diverges while the area stays exactly $1$. That limiting
object, defined by the sifting property
$\int f(x)\,\delta(x)\,dx = f(0)$, is the delta; in three dimensions
$\rho(\mathbf r) = q\,\delta^3(\mathbf r - \mathbf r_0)$.

**This exercise.** For $w = 1, 0.3, 0.1, 0.03$:

1. Confirm with `numpy.trapezoid` that $\int g_w\,dx = 1$ at every width
   while the peak grows as $1/w$.
2. Plot the four Gaussians superposed on one axes
   ({numref}`fig-coulomb-delta`): the narrowing curves show the diverging
   peak and the constant unit area at a glance.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    areas,
    np.ones_like(areas),
    "every nascent-delta Gaussian has unit integral as it narrows",
    rtol=1e-3,
)
validate.close(
    peaks * widths,
    (1 / np.sqrt(2 * np.pi)) * np.ones_like(widths),
    "the nascent-delta peak grows as 1/w",
    rtol=1e-6,
)

## Exercise 6 — Gauss's law previewed: flux of a point charge (student exercise)

Here is the integral that organises all of
electrostatics, and it is yours to compute. The **flux** of $\mathbf E$ through a
closed surface, $\oint \mathbf E\cdot d\mathbf A$, measures the field streaming
outward through it. For a point charge the inverse-square falloff of the field and
the $r^2$ growth of a sphere's area cancel exactly, so the flux equals
$q/\varepsilon_0$ *regardless of the sphere's radius*. That radius-independence is
the integral signature of the delta source and the gateway to Gauss's law
([§3.3](gauss-law.ipynb)).

**This exercise.** For $q = 1\,$nC at the origin and concentric spheres of
radius $R = 0.2, 0.5, 1.0\,$m:

1. Compute $\oint \mathbf E\cdot d\mathbf A$ numerically with
   `numpy.trapezoid` over $(\theta, \varphi)$.
2. Confirm it equals $q/\varepsilon_0$ at every radius.

Because the check tests the *flux*, a ✗ means "re-examine the surface integral or
the field," never a plotting issue.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    flux / expected,
    np.ones_like(flux),
    "the flux through any enclosing sphere is q/ε₀, independent of radius",
    rtol=1e-3,
)

## Exercise 7 — A continuous distribution: the charged ring

Real sources are continuous, and superposition
{eq}`eq-superposition` handles them by integration. A uniformly charged ring is
the cleanest case: by symmetry its on-axis field is purely axial and has a tidy
closed form, $E_z = kQz/(z^2+R^2)^{3/2}$, against which we can check a numerical
integral over the distribution.

**This exercise.** Take a ring of radius $R = 0.05\,$m carrying total charge
$Q = 1\,$nC in the $z=0$ plane.

1. Compute the on-axis field $E_z(z)$ by integrating over the ring
   (`numpy.trapezoid` in the azimuth).
2. Confirm it matches the closed form $E_z = kQz/(z^2+R^2)^{3/2}$.

{numref}`fig-coulomb-ring` shows the field in a plane through the axis, built
by superposing many point charges around the ring.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    Ez_num,
    Ez_closed,
    "the numerically integrated ring field matches the closed form",
    rtol=1e-4,
)

## Exercise 8 — Numerical divergence of E (synthesis, forward to 3.3)

One last tool, introduced briefly and developed
fully in [§3.3](gauss-law.ipynb). Gauss's law has a local form, $\nabla\cdot\mathbf E = \rho/
\varepsilon_0$, which says the field has *sources* exactly where charge sits and is
**divergence-free** in empty space. We can compute $\nabla\cdot\mathbf E$
numerically from the field samples with finite differences and watch it vanish
away from the charge.

**This exercise.** On a 3-D grid that avoids the origin:

1. Evaluate the point-charge field and take its numerical divergence with
   `numpy.gradient`.
2. Confirm it is negligible compared with the size of its own derivative
   terms once we step away from the source.

The spike of divergence *at* the charge is the delta of {eq}`eq-delta`, made
precise in [§3.3](gauss-law.ipynb).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    rel_div < 0.05,
    "the point-charge field is divergence-free in empty space (∇·E ≈ 0 away from the source)",
    f"RMS relative divergence in the empty-space shell = {rel_div:.3e}",
)

## Notebook summary

- The point-charge field $\mathbf E=kq\hat{\mathbf r}/r^2$ and its inverse-square fall
  ($E(0.1)/E(0.2)=4$), with superposition cancelling components by symmetry.
- The dipole's on-axis field $\propto 1/r^3$ ($E(0.1)/E(0.2)=8$); field lines traced from
  $d\mathbf r/ds=\mathbf E/|\mathbf E|$ and seeded at equal angles, so the line density
  faithfully encodes $|\mathbf E|$.
- The Dirac delta built as a unit-area Gaussian sequence; the flux of a point charge
  integrating to $q/\varepsilon_0$ (a Gauss's-law preview); the charged ring's on-axis
  field; and $\nabla\cdot\mathbf E$ taken numerically with `numpy.gradient`, the local
  law developed in [§3.3](gauss-law.ipynb).
- Tools introduced: `ecp.draw.radial_field_lines` and `ecp.draw.field_quiver` for
  physically faithful field plots, never a `streamplot` density artefact.

## Outlook

- **More elaborate distributions.** The same superposition integral handles
  surfaces and volumes; solid-angle arguments often replace the integral with a
  geometric picture.
- **Earnshaw's theorem.** Because $\nabla\cdot\mathbf E = 0$ in empty space, no
  arrangement of fixed charges can trap another charge in stable equilibrium, a
  clean impossibility result that echoes the saddle-point arguments of
  [§2.7](../02-classical-mechanics/small-oscillations.ipynb).
- **Derivatives of the delta.** The ideal dipole is $-\mathbf p\cdot\nabla\delta$,
  the first in a tower of distributional derivatives that organise the multipole
  expansion of [§3.5](multipole-expansion.ipynb).
- **Forward links.** The potential ([§3.2](potential-energy.ipynb)) trades this
  vector field for a scalar; Gauss's law ([§3.3](gauss-law.ipynb)) makes the flux
  of this notebook a law; and the Green's-function solution of Poisson's
  equation ([§3.4](laplace-poisson.ipynb)) rests on
  $\nabla^2(1/r) = -4\pi\,\delta^3(\mathbf r)$, which makes the point charge the
  fundamental solution of electrostatics.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()